In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("cs-training.csv")
print(df.shape)
df.head()

(150000, 12)


,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [2]:
# MonthlyIncome: impute with median (income is skewed, median is more robust than mean)
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())

# NumberOfDependents: impute with 0 (most common value; absence likely means none)
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)

# Cap extreme outliers (age=0 and utilization > 10 are known data errors in this dataset)
df = df[(df['age'] > 0)]
df['RevolvingUtilizationOfUnsecuredLines'] = df['RevolvingUtilizationOfUnsecuredLines'].clip(upper=2)

In [3]:
# Total number of past-due incidents across all severity buckets
df['TotalPastDueIncidents'] = (
    df['NumberOfTime30-59DaysPastDueNotWorse'] +
    df['NumberOfTime60-89DaysPastDueNotWorse'] +
    df['NumberOfTimes90DaysLate']
)

# Income-to-debt style ratio bucket for the dashboard
df['DebtRatioBucket'] = pd.cut(df['DebtRatio'],
    bins=[-0.01, 0.2, 0.4, 0.6, 1, np.inf],
    labels=['Low (<20%)', 'Moderate (20-40%)', 'Elevated (40-60%)', 'High (60-100%)', 'Very High (>100%)'])

# Age band for segmentation in Power BI
df['AgeBand'] = pd.cut(df['age'], bins=[18,30,40,50,60,70,100],
    labels=['18-29','30-39','40-49','50-59','60-69','70+'])

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

features = ['RevolvingUtilizationOfUnsecuredLines','age','TotalPastDueIncidents',
            'DebtRatio','MonthlyIncome','NumberOfOpenCreditLinesAndLoans',
            'NumberRealEstateLoansOrLines','NumberOfDependents']

X = df[features]
y = df['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:,1]))
print(classification_report(y_test, model.predict(X_test)))

# Score EVERY customer (not just the test set) — this is what powers the dashboard
df['RiskProbability'] = model.predict_proba(X[features])[:,1]
df['RiskTier'] = pd.cut(df['RiskProbability'], bins=[-0.01,0.2,0.5,1],
    labels=['Low Risk','Medium Risk','High Risk'])

AUC: 0.8346427198658115
              precision    recall  f1-score   support

           0       0.98      0.77      0.86     27995
           1       0.18      0.74      0.30      2005

    accuracy                           0.76     30000
   macro avg       0.58      0.75      0.58     30000
weighted avg       0.92      0.76      0.82     30000



/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [5]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/customer_risk_scored.csv', index=False)
print("Saved:", df.shape)

Saved: (149999, 17)


In [6]:
import os
print(os.path.exists('/data/processed/customer_risk_scored.csv'))

True


In [7]:
from google.colab import files
files.download('/data/processed/customer_risk_scored.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import requests

# Get the public IP address of the Colab runtime
ip_address = requests.get('https://api.ipify.org').text
print(f"Colab Public IP Address: {ip_address}")

Colab Public IP Address: 34.158.98.69


In [11]:
!curl https://packages.microsoft.com/keys/microsoft.asc | apt-key add -
!curl https://packages.microsoft.com/config/ubuntu/20.04/prod.list > /etc/apt/sources.list.d/mssql-release.list
!apt-get update
!ACCEPT_EULA=Y apt-get install -y msodbcsql18
!pip install pyodbc sqlalchemy
import sqlalchemy
import urllib

server = 'bankrisk-sqlserver-2026.database.windows.net'
database = 'free-sql-db-5661342'   # your actual database name, not "BankRiskDB"
username = 'CloudSA0b33a45f'       # your SQL server admin login
password = 'Shesh@739713' # whatever you last reset it to
driver = '{ODBC Driver 18 for SQL Server}'

params = urllib.parse.quote_plus(
    f'DRIVER={driver};SERVER={server};DATABASE={database};UID={username};PWD={password}'
)
engine = sqlalchemy.create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

df_final = df.reset_index().rename(columns={'index': 'CustomerID', 'age': 'Age'})
cols = ['CustomerID','SeriousDlqin2yrs','RevolvingUtilizationOfUnsecuredLines','Age','AgeBand',
        'TotalPastDueIncidents','DebtRatio','DebtRatioBucket','MonthlyIncome',
        'NumberOfOpenCreditLinesAndLoans','NumberRealEstateLoansOrLines','NumberOfDependents',
        'RiskProbability','RiskTier']

df_final[cols].to_sql('CustomerRisk', engine, if_exists='replace', index=False, chunksize=1000)
print("Upload complete.")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0Warning: apt-key is deprecated. Manage keyring files in trusted.gpg.d instead (see apt-key(8)).
100   975  100   975    0     0   6416      0 --:--:-- --:--:-- --:--:--  6456
OK
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    89  100    89    0     0    878      0 --:--:-- --:--:-- --:--:--   881
Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:2 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 https://cli.github.com/packages stable InRelease
Hit:6 https://packages.microsoft.com/ubuntu/20

In [12]:
check = pd.read_sql("SELECT COUNT(*) AS row_count FROM CustomerRisk", engine)
print(check)

   row_count
0     149999


In [13]:
from dotenv import load_dotenv
import os
load_dotenv()
password = os.getenv('Shesh739713')